# Kinyarwanda ASR demo: MMS-1B + fine-tuned adapter

Runs [`facebook/mms-1b-all`](https://huggingface.co/facebook/mms-1b-all) with the
fine-tuned Kinyarwanda adapter
[`zoro6u/mms-1b-kinyarwanda-adapter-3k`](https://huggingface.co/zoro6u/mms-1b-kinyarwanda-adapter-3k)
and transcribes one audio file. No Hugging Face Space needed: runs locally (CPU is
fine), on Colab or on Kaggle.

- **Download:** ~3.9 GB base model on first run (cached afterwards) + 8.6 MB adapter.
- **Memory:** the model is ~965M parameters; plan for roughly 4-5 GB of free RAM.
- **Accuracy:** 45.5% WER on the 323-sentence FLEURS-style dev set
  (`results/run_meta.json`), i.e. roughly every second word is wrong.
- **Licence:** CC-BY-NC-4.0 (inherited from the base model), non-commercial use only.

The model loading follows the Usage section of `MODEL_CARD.md` step by step. The only
difference is audio decoding: this notebook uses ffmpeg (through `transformers`) instead
of `librosa`, so it needs the `ffmpeg` binary but not librosa. It decodes any format
ffmpeg reads and resamples to 16 kHz.

In [ ]:
# Torch is assumed to be installed already (it is on Colab/Kaggle). Locally, for CPU:
#   pip install torch --index-url https://download.pytorch.org/whl/cpu
%pip install -q transformers==5.16.1 safetensors huggingface_hub

In [ ]:
import shutil

import torch
import transformers

assert transformers.__version__ == "5.16.1", f"expected transformers 5.16.1, got {transformers.__version__}"
assert shutil.which("ffmpeg"), "ffmpeg not found (apt install ffmpeg / dnf install ffmpeg-free)"
print("torch", torch.__version__, "| transformers", transformers.__version__)

## Load the model

Same sequence as `MODEL_CARD.md`: load the base model with the pretrained `kin`
adapter (the starting point of fine-tuning), then overwrite the adapter tensors with
the fine-tuned ones.

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoProcessor, Wav2Vec2ForCTC

BASE = "facebook/mms-1b-all"
ADAPTER = "zoro6u/mms-1b-kinyarwanda-adapter-3k"

processor = AutoProcessor.from_pretrained(BASE, target_lang="kin")
model = Wav2Vec2ForCTC.from_pretrained(BASE, target_lang="kin", ignore_mismatched_sizes=True)
model.load_adapter("kin")                       # the pretrained starting point

path = hf_hub_download(ADAPTER, "adapter.kin.safetensors")
result = model.load_state_dict(load_file(path), strict=False)
assert not result.unexpected_keys               # only adapter tensors were saved
model.eval()

## Transcribe

Set `AUDIO_PATH` to your file (wav, mp3, m4a, ... anything ffmpeg can read).

In [ ]:
from transformers.pipelines.audio_utils import ffmpeg_read

AUDIO_PATH = "clip.wav"

with open(AUDIO_PATH, "rb") as f:
    audio = ffmpeg_read(f.read(), 16000)        # mono float32 at 16 kHz
print(f"{len(audio) / 16000:.1f} s of audio")

inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
    ids = torch.argmax(model(**inputs).logits, dim=-1)[0]
print(processor.decode(ids))